# Suraksha Setu — complaint CNN training (doc 06 tasks 4B.2–4B.3)

Runtime → Change runtime type → **GPU** (T4). Raw photos live in the team Google Drive as
`SurakshaSetu/raw/<source>/<class>/*.jpg` — never in git. See `ml/README.md`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/SurakshaSetu"
VERSION = 1  # bump for every released model

In [ ]:
!git clone --depth 1 https://github.com/Ritik0712-ai/suraksha_setu.git
%cd suraksha_setu
!pip install -q ai-edge-litert scikit-learn

## 1. Prepare the dataset
Check the per-class counts it prints: target ≥ 300 per class.

In [ ]:
!python ml/prepare_dataset.py --out data/v{VERSION} --sources \
    {DRIVE}/raw/rdd2022_india {DRIVE}/raw/open_images {DRIVE}/raw/field_visit_1
!cat data/v{VERSION}/manifest.json

## 2. Train (MobileNetV2, 2 phases)

In [ ]:
!python ml/train_cnn.py --data data/v{VERSION} --out runs/v{VERSION}

## 3. Evaluate
Target: test accuracy ≥ 0.87. Paste the accuracy, per-class F1 and the confusion matrix into the review.

In [ ]:
!python ml/evaluate.py --model runs/v{VERSION}/best.keras --data data/v{VERSION} --out runs/v{VERSION}
import pandas as pd
pd.read_csv(f"runs/v{VERSION}/confusion_matrix.csv", index_col=0)

## 4. Export to .tflite (float16) and save to Drive
Then publish both files as a GitHub Release `model-v{VERSION}` (see `ml/README.md` §3).

In [ ]:
!python ml/export_tflite.py --model runs/v{VERSION}/best.keras --data data/v{VERSION} \
    --version {VERSION} --metrics runs/v{VERSION}/metrics.json --out dist
!mkdir -p {DRIVE}/models/v{VERSION} && cp dist/* runs/v{VERSION}/*.json runs/v{VERSION}/*.csv {DRIVE}/models/v{VERSION}/
!ls -la {DRIVE}/models/v{VERSION}